# Bangkok CCTV Vehicle Detection — 2110531 DSDE Take-Home Midterm (2026/1)

End-to-end pipeline: **EDA → data cleaning → camera-held-out split → YOLO fine-tuning → mAP@50 evaluation (pycocotools, same as Kaggle) → visualization → submission CSV**.

* Kaggle: https://www.kaggle.com/competitions/2110531-dsde-2026-1
* Metric: COCO-style mAP@50 (mean AP over 8 classes)
* Run this notebook from the `midterm/` folder. Raw data is expected in `data/raw/` (`kaggle competitions download -c 2110531-dsde-2026-1 -p data/raw` then unzip).

**Key facts found in EDA (see below):**
1. Test cameras (5) are **disjoint** from train cameras (15) → validation must hold out whole cameras, otherwise the val score is over-optimistic.
2. Images are tiny (352×288) and boxes are small (median ≈ 13×20 px) → upscale (`imgsz` ≥ 640).
3. Strong class imbalance (Car 17.7k boxes vs Songthaew 117) and mAP weights every class equally → repeat-factor sampling for rare classes.

## 0. Setup

In [ ]:
# Environment used: Python 3.12, torch 2.8.0+cu126, ultralytics 8.4.163 (see requirements.txt)
# %pip install -r requirements.txt
import os, re, json, shutil, random, itertools, math, contextlib, io
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch
import ultralytics
from ultralytics import YOLO, RTDETR

print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('ultralytics', ultralytics.__version__)

### Config
Change the values here only. Model choices (all COCO-pretrained, loaded automatically by ultralytics):

| Family | Weights | Note |
|---|---|---|
| **YOLO26** (2026) | `yolo26n/s/m/l/x.pt` | newest, end-to-end NMS-free (covered in the Week 5 slides) |
| YOLO11 (2024) | `yolo11n/s/m/l/x.pt` | strong, well-tested baseline |
| YOLOv8 | `yolov8n/s/m/l/x.pt` | model used in the course example notebook |
| RT-DETR | `rtdetr-l.pt`, `rtdetr-x.pt` | transformer detector, heavier — needs a bigger GPU |

With a 4 GB laptop GPU: `s` @ 640 with batch 8 fits. For `m`/`l` or `imgsz` ≥ 960 use Colab / Kaggle GPUs.

In [ ]:
ROOT     = Path.cwd().resolve()            # .../midterm
RAW      = ROOT / 'data' / 'raw'
YOLO_DIR = ROOT / 'data' / 'yolo'          # generated YOLO-format dataset
RUNS     = ROOT / 'runs'
SUB_DIR  = ROOT / 'submissions'
CLASSES  = ['Car', 'Motorcycle', 'Bus', 'Truck', 'Tuktuk', 'Van', 'Pickup', 'Songthaew']

# every value below can also be overridden by an environment variable of the same name (for batch runs)
env = os.environ.get
SMOKE_TEST = env('SMOKE_TEST') == '1'   # quick pipeline check: 1 epoch on 10% of data

# ---- model / training ----
MODEL      = env('MODEL', 'yolo26s.pt')
IMGSZ      = int(env('IMGSZ', 640))
EPOCHS     = 1 if SMOKE_TEST else int(env('EPOCHS', 100))
BATCH      = int(env('BATCH', 8))
PATIENCE   = 30
WORKERS    = 2
SEED       = 42
FULL_TRAIN = env('FULL_TRAIN', '0') == '1'   # 0: train on train-cams, validate on held-out cams. 1: train on all 15 cams (final model)
SKIP_TRAIN = env('SKIP_TRAIN', '0') == '1'   # 1: skip training and load WEIGHTS (for reproducing results / TTA only)
WEIGHTS    = Path(env('WEIGHTS')) if env('WEIGHTS') else None   # e.g. weights/best.pt when SKIP_TRAIN=1

# ---- data prep ----
N_VAL_CAMS = 4          # held-out cameras for validation (test set has 5 unseen cameras)
MIN_BOX    = 2          # drop boxes narrower/shorter than this (px) from TRAINING labels
USE_RFS    = True       # repeat-factor sampling for rare classes
RFS_T      = 0.15       # images containing a class seen in < RFS_T of images get repeated
RFS_MAX    = 4

# ---- inference ----
CONF    = 0.001         # keep low-confidence boxes: mAP is not penalised by them
MAX_DET = 100           # pycocotools only uses the top-100 detections per image
TTA     = env('TTA', '0') == '1'

RUN_NAME = env('RUN_NAME') or f"{Path(MODEL).stem}_{IMGSZ}" + ('_full' if FULL_TRAIN else '') + ('_smoke' if SMOKE_TEST else '')

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(SEED)
for d in (RUNS, SUB_DIR):
    d.mkdir(parents=True, exist_ok=True)
print(RUN_NAME)

## 1. Load data

In [ ]:
train_df    = pd.read_csv(RAW / 'train.csv')
sample_sub  = pd.read_csv(RAW / 'sample_submission.csv')
TRAIN_IMG_DIR = RAW / 'train' / 'train'
TEST_IMG_DIR  = RAW / 'test' / 'test'
train_files = sorted(os.listdir(TRAIN_IMG_DIR))
test_files  = sorted(os.listdir(TEST_IMG_DIR))

def parse_name(f):
    cam, date, time = Path(f).stem.split('_')
    return cam, pd.to_datetime(date + time, format='%Y%m%d%H%M%S')

meta = pd.DataFrame([(f, 'train', *parse_name(f)) for f in train_files] +
                    [(f, 'test',  *parse_name(f)) for f in test_files],
                    columns=['file', 'split', 'cam', 'time'])
meta['hour'] = meta.time.dt.hour + meta.time.dt.minute / 60
sizes = {f: Image.open((TRAIN_IMG_DIR if s == 'train' else TEST_IMG_DIR) / f).size for f, s in zip(meta.file, meta.split)}
meta['w'] = meta.file.map(lambda f: sizes[f][0]); meta['h'] = meta.file.map(lambda f: sizes[f][1])

train_df['cam'] = train_df.image_id.str.split('_').str[0]
train_df['bw'] = train_df.x2 - train_df.x1
train_df['bh'] = train_df.y2 - train_df.y1

print('train rows:', train_df.shape, '| train images:', len(train_files), '| test images:', len(test_files))
print('image sizes:', Counter(zip(meta.w, meta.h)))
train_df.head()

### Note on `sample_submission.csv`
`sample_submission.csv` uses long ids such as `dataset_1068_annotated_coco1.0_1068_<Thai location>_20260825_060110.jpg`,
while the test images are named `1068_20260825_060110.jpg`. The Kaggle solution uses the **file names**
(verified by submission: long ids → 0.000, file names → 0.516), so the submission is written with file names (Section 8).
The check below only confirms that every long id corresponds to a test file.

In [ ]:
def sub_key(long_id):
    cam = re.match(r'dataset_(\d+)_', long_id).group(1)
    ts = re.search(r'(\d{8}_\d{6})\.jpg$', long_id).group(1)
    return f'{cam}_{ts}'

key2subid = {sub_key(s): s for s in sample_sub.image_id}
assert len(key2subid) == sample_sub.image_id.nunique()
test_keys = {Path(f).stem for f in test_files}
print('sample_submission images:', len(key2subid), '| all mapped to a test file:', set(key2subid) <= test_keys)
print('test images not listed in sample_submission:', len(test_keys - set(key2subid)), '(still predicted and submitted)')
sample_sub.head(3)

## 2. EDA
Follows the Data-Preparation steps of the course (Week 2 slides / Lab1–Lab4):
**(1) examine the data set** → preview, data dictionary, missing values, out-of-range & miscodes, target distribution;
**(2) narrow down / understand features** → derived features from the file name, association with the target (chi-square), leakage;
**(3) outliers & skewness**; **(4) train / validation split** (Section 4).
Figures are also saved to `report/figures/` for the report.

### 2.1 Preview, data dictionary and missing values

In [ ]:
FIG_DIR = ROOT / 'report' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
def save(name):
    plt.savefig(FIG_DIR / f'{name}.png', dpi=110, bbox_inches='tight')

PALETTE = [(255, 56, 56), (255, 157, 151), (255, 112, 31), (255, 178, 29), (207, 210, 49), (72, 249, 10), (26, 147, 52), (0, 212, 187)]

def draw(img_path, boxes, conf_thr=None):
    # boxes: DataFrame with class_id, x1..y2 (+ optional confidence)
    im = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    im = cv2.resize(im, None, fx=2, fy=2, interpolation=cv2.INTER_LINEAR)   # upscale for readability
    for r in boxes.itertuples():
        if conf_thr is not None and r.confidence < conf_thr:
            continue
        c = PALETTE[int(r.class_id)]
        p1, p2 = (int(r.x1 * 2), int(r.y1 * 2)), (int(r.x2 * 2), int(r.y2 * 2))
        cv2.rectangle(im, p1, p2, c, 2)
        label = CLASSES[int(r.class_id)] + (f' {r.confidence:.2f}' if hasattr(r, 'confidence') else '')
        cv2.putText(im, label, (p1[0], max(p1[1] - 3, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, c, 1, cv2.LINE_AA)
    return im

def show_grid(items, ncols=4, size=4.5, name=None):
    nrows = math.ceil(len(items) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * size, nrows * size * 0.82))
    for ax in np.ravel(axes):
        ax.axis('off')
    for ax, (im, title) in zip(np.ravel(axes), items):
        ax.imshow(im); ax.set_title(title, fontsize=9)
    plt.tight_layout()
    if name:
        save(name)
    plt.show()

RAW_COLS = ['id', 'image_id', 'class_id', 'x1', 'y1', 'x2', 'y2']
data_dict = pd.DataFrame([
    ('id', 'int', 'row id - identifier only, not used as a feature'),
    ('image_id', 'str', 'image file = <camera>_<YYYYMMDD>_<HHMMSS>.jpg -> derived: camera, date, hour, day/night'),
    ('class_id', 'int (nominal)', 'target: 0 Car, 1 Motorcycle, 2 Bus, 3 Truck, 4 Tuktuk, 5 Van, 6 Pickup, 7 Songthaew'),
    ('x1, y1', 'int', 'top-left corner of the box [px]'),
    ('x2, y2', 'int', 'bottom-right corner of the box [px]'),
], columns=['column', 'type', 'description'])
display(data_dict)
print('train.csv shape:', train_df[RAW_COLS].shape)
display(pd.DataFrame({'dtype': train_df[RAW_COLS].dtypes.astype(str),
                      'n_unique': train_df[RAW_COLS].nunique(),
                      'missing': train_df[RAW_COLS].isnull().sum()}))
display(train_df[RAW_COLS].describe().T.round(2))
print('image-level table (features derived from the file name):')
meta.head()

### 2.2 Out-of-range values and miscodes

In [ ]:
W_ = train_df.image_id.map(lambda f: sizes.get(f, (np.nan, np.nan))[0])
H_ = train_df.image_id.map(lambda f: sizes.get(f, (np.nan, np.nan))[1])
checks = {
    'class_id outside 0-7': int((~train_df.class_id.between(0, 7)).sum()),
    'x1 >= x2 or y1 >= y2 (inverted / zero box)': int(((train_df.x1 >= train_df.x2) | (train_df.y1 >= train_df.y2)).sum()),
    'box outside the image': int(((train_df.x1 < 0) | (train_df.y1 < 0) | (train_df.x2 > W_) | (train_df.y2 > H_)).sum()),
    'image_id not found on disk': int((~train_df.image_id.isin(train_files)).sum()),
    'duplicated row id': int(train_df.id.duplicated().sum()),
    'exact duplicate boxes': int(train_df.duplicated(['image_id', 'class_id', 'x1', 'y1', 'x2', 'y2']).sum()),
    'boxes with w or h < 2 px': int(((train_df.bw < 2) | (train_df.bh < 2)).sum()),
    'train images without any box (background)': len(set(train_files) - set(train_df.image_id)),
    'images not 352x288': sum(s != (352, 288) for s in sizes.values()),
}
pd.Series(checks, name='count').to_frame()

### 2.3 Target distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
cnt = train_df.class_id.value_counts().sort_index()
ax = sns.barplot(x=[CLASSES[i] for i in cnt.index], y=cnt.values, ax=axes[0], color='steelblue')
ax.bar_label(ax.containers[0]); ax.set_title('Boxes per class (train, log scale)'); ax.set_yscale('log')
img_frac = train_df.groupby('class_id').image_id.nunique() / len(train_files)
ax = sns.barplot(x=[CLASSES[i] for i in img_frac.index], y=img_frac.values, ax=axes[1], color='darkorange')
ax.bar_label(ax.containers[0], fmt='%.3f'); ax.set_title('Fraction of train images containing class')
plt.tight_layout(); save('eda_class_dist'); plt.show()
dist = pd.DataFrame({'boxes': cnt.values, 'share %': (cnt / cnt.sum() * 100).round(2).values,
                     'images': train_df.groupby('class_id').image_id.nunique().values}, index=CLASSES)
dist['imbalance vs Car'] = (dist.boxes.iloc[0] / dist.boxes).round(1)
dist

### 2.4 Camera vs class — association test (chi-square)
H0: the class distribution is the same for every camera. A significant result means the camera (viewpoint / road type)
changes which vehicles are seen → a model can over-fit to camera-specific patterns, which matters because the **test cameras are unseen**.

In [ ]:
from scipy.stats import chi2_contingency
fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={'width_ratios': [1, 1.4]})
cams_count = meta.groupby(['cam', 'split']).size().unstack(fill_value=0)
cams_count.plot.bar(stacked=True, ax=axes[0], title='Images per camera (train cams and test cams are disjoint!)')
cam_cls = pd.crosstab(train_df.cam, train_df.class_id).reindex(columns=range(8), fill_value=0)
cam_cls.columns = CLASSES
sns.heatmap(cam_cls, annot=True, fmt='d', cmap='Blues', norm=plt.matplotlib.colors.LogNorm(), ax=axes[1], cbar=False)
axes[1].set_title('Boxes per camera x class (train)')
plt.tight_layout(); save('eda_camera'); plt.show()
print('train cams:', sorted(meta[meta.split == 'train'].cam.unique()))
print('test  cams:', sorted(meta[meta.split == 'test'].cam.unique()))

chi2, p, dof, expected = chi2_contingency(cam_cls.values)
cramers_v = np.sqrt(chi2 / (cam_cls.values.sum() * (min(cam_cls.shape) - 1)))
print(f'chi-square camera x class: chi2 = {chi2:,.0f}, dof = {dof}, p = {p:.1e}, Cramer\'s V = {cramers_v:.3f}')
print(f'cells with expected count < 5: {(expected < 5).mean():.0%} (rare classes)')
resid = pd.DataFrame((cam_cls.values - expected) / np.sqrt(expected), index=cam_cls.index, columns=CLASSES)
fig, ax = plt.subplots(figsize=(10, 5.5))
sns.heatmap(resid.clip(-15, 15), annot=resid.round(0).astype(int), fmt='d', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Standardised residuals (observed - expected) / sqrt(expected): red = more than expected')
plt.tight_layout(); save('eda_chi2_resid'); plt.show()

### 2.5 Box size — outliers and skewness (as in Lab4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
train_df.groupby('image_id').size().reindex(train_files, fill_value=0).plot.hist(bins=48, ax=axes[0], title='Boxes per image')
train_df['size'] = np.sqrt(train_df.bw * train_df.bh)
sns.boxplot(x=train_df.class_id.map(dict(enumerate(CLASSES))), y=train_df['size'], ax=axes[1], order=CLASSES)
axes[1].set_title('Box size sqrt(w*h) [px] per class'); axes[1].tick_params(axis='x', rotation=30)
axes[2].scatter(train_df.bw, train_df.bh, s=1, alpha=.2); axes[2].set_xlabel('w'); axes[2].set_ylabel('h'); axes[2].set_title('Box w vs h [px]')
plt.tight_layout(); save('eda_box_size'); plt.show()
print(train_df[['bw', 'bh', 'size']].describe().T.round(1))

In [ ]:
from scipy.stats import skew, kurtosis
train_df['area'] = train_df.bw * train_df.bh
train_df['log_area'] = np.log(train_df.area)
train_df['aspect'] = train_df.bh / train_df.bw
fig, axes = plt.subplots(1, 2, figsize=(15, 3.8))
for ax, col, name in [(axes[0], 'area', 'box area [px^2]'), (axes[1], 'log_area', 'log(box area)')]:
    s = train_df[col]
    s.plot.hist(bins=100, ax=ax, color='slategray')
    ax.set_title(f'{name}: skew = {skew(s):.2f}, kurtosis = {kurtosis(s):.2f}')
plt.tight_layout(); save('eda_area_skew'); plt.show()

def iqr_outlier(s):
    q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    return (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)

rows = []
for c, g in train_df.groupby('class_id'):
    rows.append((CLASSES[c], len(g), round(g.area.median()), int(iqr_outlier(g.log_area).sum()),
                 round(g.aspect.median(), 2), int(iqr_outlier(g.aspect).sum()),
                 int((np.abs(g.log_area - g.log_area.mean()) > 3 * g.log_area.std()).sum())))
outliers = pd.DataFrame(rows, columns=['class', 'boxes', 'median area', 'log-area IQR outliers', 'median h/w',
                                       'h/w IQR outliers', 'log-area > 3 std'])
display(outliers)

# look at the most extreme boxes before deciding what to do with them
ext = pd.concat([train_df.nlargest(4, 'area'), train_df.nlargest(4, 'aspect'), train_df.nsmallest(4, 'aspect')])
items = [(draw(TRAIN_IMG_DIR / r.image_id, ext.loc[[i]]), f'{CLASSES[r.class_id]} {r.bw}x{r.bh}px (h/w {r.aspect:.1f})')
         for i, r in ext.iterrows()]
show_grid(items, ncols=4, size=3.6, name='eda_extreme_boxes')

**Decision:** box area is strongly right-skewed (skew 7.05, kurtosis 96 → skew 0.19 after log), which is why the images are **upscaled** and the
detector works on a multi-scale feature pyramid. Inspecting the extremes (figure above):
* the **largest boxes are real vehicles** very close to camera 1426 (6 boxes are almost full-height) → **kept**, the test set can contain the same;
* the **extreme aspect ratios are annotation errors** — slivers such as 1×8 or 2×16 px with no vehicle inside
  (195 boxes have a side ≤ 2 px; 10 have h/w > 6 or < 1/6). Boxes with a side < 2 px are removed in Section 3;
  a stricter filter is left as future work so that the trained models stay reproducible.
* Capping/truncating outliers (Lab4) is not meaningful for box coordinates, so no truncation is applied.

### 2.6 Time of day — brightness, traffic density, day vs night

In [ ]:
meta['brightness'] = [np.asarray(Image.open((TRAIN_IMG_DIR if s == 'train' else TEST_IMG_DIR) / f).convert('L')).mean()
                      for f, s in zip(meta.file, meta.split)]
fig, ax = plt.subplots(figsize=(10, 4))
sns.scatterplot(data=meta, x='hour', y='brightness', hue='split', s=10, alpha=.5, ax=ax)
ax.set_title('Mean brightness vs hour of day'); save('eda_brightness'); plt.show()
# street lights keep night frames bright, so brightness alone can't separate them -> use clock time (sunset ~18:30)
meta['night'] = meta.hour >= 18.5
print(meta.groupby('split').night.mean().rename('night fraction'))

In [ ]:
night_of = dict(zip(meta.file, meta.night))
train_df['night'] = train_df.image_id.map(night_of)
tr_meta = meta[meta.split == 'train'].copy()
tr_meta['boxes'] = tr_meta.file.map(train_df.groupby('image_id').size()).fillna(0)
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
tr_meta.groupby(tr_meta.hour.astype(int)).boxes.mean().plot.bar(ax=axes[0], color='teal', title='Mean boxes per image by hour (traffic density)')
share = pd.crosstab(train_df.night.map({False: 'day', True: 'night'}), train_df.class_id.map(dict(enumerate(CLASSES))), normalize='index')[CLASSES]
(share.T * 100).plot.bar(ax=axes[1], title='Class share [%]: day vs night', logy=True)
plt.tight_layout(); save('eda_time'); plt.show()
tab = pd.crosstab(train_df.night, train_df.class_id)
chi2_n, p_n, dof_n, _ = chi2_contingency(tab.values)
print(f'chi-square day/night x class: chi2 = {chi2_n:.0f}, dof = {dof_n}, p = {p_n:.1e}, Cramer\'s V = {np.sqrt(chi2_n / tab.values.sum()):.3f}')
(share * 100).round(2)

### 2.7 Leakage check — how similar are frames of the same camera?
In Lab1, columns that "leak" information were removed. Here the leak is at the **image** level: frames of one camera, 30 minutes apart,
share the same background and viewpoint. If such frames are split randomly between train and validation, validation measures memorisation
of the scene, not generalisation to a new camera (which is what the test set requires).

In [ ]:
def thumb(p):
    return np.asarray(Image.open(p).convert('L').resize((88, 72)), dtype=np.float32)
trs = tr_meta.sort_values(['cam', 'time'])
th = {f: thumb(TRAIN_IMG_DIR / f) for f in trs.file}
same = [np.abs(th[a] - th[b]).mean() for _, g in trs.groupby('cam') for a, b in zip(g.file.values[:-1], g.file.values[1:])]
rs = np.random.default_rng(SEED); fa, ca = trs.file.values, trs.cam.values
cross = []
while len(cross) < len(same):
    i, j = rs.integers(len(fa), size=2)
    if ca[i] != ca[j]:
        cross.append(np.abs(th[fa[i]] - th[fa[j]]).mean())
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.hist(same, bins=60, alpha=.7, label=f'same camera, consecutive frames (median {np.median(same):.1f})')
ax.hist(cross, bins=60, alpha=.7, label=f'different cameras (median {np.median(cross):.1f})')
ax.set_xlabel('mean absolute pixel difference (grayscale 88x72)'); ax.legend(); ax.set_title('Frame similarity')
save('eda_leakage'); plt.show()
print(f'same-camera frames differ {np.median(cross) / np.median(same):.1f}x less than frames of different cameras')

### 2.8 Train vs test distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 3.8))
sns.histplot(meta, x='brightness', hue='split', stat='density', common_norm=False, bins=40, ax=axes[0])
axes[0].set_title('Brightness: train vs test')
sns.histplot(meta, x='hour', hue='split', stat='density', common_norm=False, bins=15, ax=axes[1])
axes[1].set_title('Hour of day: train vs test')
plt.tight_layout(); save('eda_train_vs_test'); plt.show()
meta.groupby('split').agg(images=('file', 'size'), cameras=('cam', 'nunique'), night_share=('night', 'mean'),
                          brightness=('brightness', 'mean')).round(3)

### 2.9 Visual inspection

In [ ]:
gt_by_img = dict(tuple(train_df.groupby('image_id')))
empty = train_df.iloc[:0]
# preview: one random daytime frame per TRAIN camera with ground truth (training itself uses ALL frames, day and night)
rng = np.random.default_rng(SEED)
items = []
for cam, g in meta[(meta.split == 'train') & ~meta.night].groupby('cam'):
    f = g.file.iloc[rng.integers(len(g))]
    items.append((draw(TRAIN_IMG_DIR / f, gt_by_img.get(f, empty)), f'train cam {cam}: {f}'))
show_grid(items, name='eda_train_samples')
# night frames of the training set (night frames are used for training as well - only this preview is split by day/night)
night_items = []
for cam, g in meta[(meta.split == 'train') & meta.night].groupby('cam'):
    f = g.file.iloc[rng.integers(len(g))]
    night_items.append((draw(TRAIN_IMG_DIR / f, gt_by_img.get(f, empty)), f'train cam {cam} (night)'))
show_grid(night_items[:8], name='eda_train_night_samples')

In [ ]:
# TEST cameras (no labels) - check the domain shift vs train cameras; plus a few night frames
items = []
for cam, g in meta[meta.split == 'test'].groupby('cam'):
    for night in (False, True):
        gg = g[g.night == night]
        if len(gg):
            f = gg.file.iloc[rng.integers(len(gg))]
            items.append((draw(TEST_IMG_DIR / f, empty), f"test cam {cam} ({'night' if night else 'day'})"))
show_grid(items, ncols=5, size=3.8, name='eda_test_samples')

In [ ]:
# examples of the rare classes (crops)
crops = []
for c in [2, 4, 5, 6, 7, 3]:
    sub = train_df[train_df.class_id == c].sample(6, random_state=SEED)
    for r in sub.itertuples():
        im = cv2.cvtColor(cv2.imread(str(TRAIN_IMG_DIR / r.image_id)), cv2.COLOR_BGR2RGB)
        pad = 4
        crop = im[max(r.y1 - pad, 0):r.y2 + pad, max(r.x1 - pad, 0):r.x2 + pad]
        crops.append((cv2.resize(crop, (96, 96), interpolation=cv2.INTER_CUBIC), CLASSES[c]))
show_grid(crops, ncols=6, size=2, name='eda_crops')

### 2.10 EDA summary → decisions
| Finding | Evidence | Decision |
|---|---|---|
| Test cameras never appear in training | 15 train cams vs 5 different test cams | validate on **held-out cameras** (Section 4) |
| Frames of one camera are near-duplicates | 2.7: same-camera frames are much more similar than cross-camera | never split frames of a camera between train/val (leakage) |
| Class distribution depends on the camera | 2.4: chi-square p ≈ 0 | camera split must keep every class in val; RFS for rare classes |
| Strong imbalance (Car : Songthaew ≈ 150 : 1) | 2.3 | repeat-factor sampling; mAP weights classes equally |
| Tiny, right-skewed box sizes | 2.5: median ≈ 13×20 px, skewed area | upscale (imgsz 640 → 960), keep real large vehicles |
| Sliver boxes are label noise | 2.5: 1×8 / 2×16 px boxes without a vehicle | drop boxes with a side < 2 px (Section 3) |
| ~17 % night frames, similar in train and test; class mix changes at night (Songthaew 0.3 % → 1.5 %) | 2.6, 2.8: chi-square p ≈ 6e-33 | report day/night mAP; HSV augmentation |
| No missing values / miscodes; 25 duplicates, 15 tiny boxes | 2.1, 2.2 | clean in Section 3 |

## 3. Data cleaning
Applied to **training labels only**. Validation is always scored against the raw labels, the same way Kaggle scores the (raw) test labels.

In [ ]:
def box_iou(a, b):
    # a: [N,4], b: [M,4] xyxy
    lt = np.maximum(a[:, None, :2], b[None, :, :2]); rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.clip(rb - lt, 0, None).prod(-1)
    area = lambda x: (x[:, 2] - x[:, 0]) * (x[:, 3] - x[:, 1])
    return inter / (area(a)[:, None] + area(b)[None] - inter + 1e-9)

clean = train_df.copy()
log = {'raw boxes': len(clean)}
assert clean.class_id.between(0, 7).all()
missing_imgs = set(clean.image_id) - set(train_files)
log['boxes on missing images'] = clean.image_id.isin(missing_imgs).sum()
clean = clean[~clean.image_id.isin(missing_imgs)]

W = clean.image_id.map(lambda f: sizes[f][0]); H = clean.image_id.map(lambda f: sizes[f][1])
clean['x1'] = clean.x1.clip(0, W); clean['x2'] = clean.x2.clip(0, W)
clean['y1'] = clean.y1.clip(0, H); clean['y2'] = clean.y2.clip(0, H)
log['boxes out of image (clipped)'] = int(((train_df.x2 > W) | (train_df.y2 > H) | (train_df.x1 < 0) | (train_df.y1 < 0)).sum())

dup = clean.duplicated(['image_id', 'class_id', 'x1', 'y1', 'x2', 'y2'])
log['exact duplicates removed'] = int(dup.sum()); clean = clean[~dup]

tiny = ((clean.x2 - clean.x1) < MIN_BOX) | ((clean.y2 - clean.y1) < MIN_BOX)
log[f'tiny boxes (<{MIN_BOX}px) removed'] = int(tiny.sum()); clean = clean[~tiny]

# near-duplicates: same image, IoU > 0.9. Same class -> drop one; different class -> label conflict (report only)
drop_idx, conflicts = [], []
for f, g in clean.groupby('image_id'):
    if len(g) < 2:
        continue
    iou = np.triu(box_iou(g[['x1', 'y1', 'x2', 'y2']].values, g[['x1', 'y1', 'x2', 'y2']].values), 1)
    for i, j in zip(*np.where(iou > 0.9)):
        if g.class_id.iloc[i] == g.class_id.iloc[j]:
            drop_idx.append(g.index[j])
        else:
            conflicts.append((f, CLASSES[g.class_id.iloc[i]], CLASSES[g.class_id.iloc[j]]))
log['near-duplicate boxes removed (same class, IoU>0.9)'] = len(set(drop_idx))
log['class conflicts (diff class, IoU>0.9) kept'] = len(conflicts)
clean = clean.drop(index=list(set(drop_idx)))
log['clean boxes'] = len(clean)
log['background images (no boxes)'] = len(set(train_files) - set(train_df.image_id))
display(pd.Series(log, name='count').to_frame())
print('conflict pairs:', Counter((a, b) for _, a, b in conflicts).most_common(10))

## 4. Camera-held-out split + YOLO format
We pick `N_VAL_CAMS` whole cameras for validation so that every class has ≥ 10 val boxes and each class's val share is as close as possible to `N_VAL_CAMS / 15`.

In [ ]:
cams = sorted(cam_cls.index)
cc = pd.crosstab(clean.cam, clean.class_id).reindex(index=cams, columns=range(8), fill_value=0)
target = N_VAL_CAMS / len(cams)
best = None
for combo in itertools.combinations(cams, N_VAL_CAMS):
    v = cc.loc[list(combo)].sum()
    if (v < 10).any():
        continue
    score = (((v / cc.sum()) - target) ** 2).sum()
    if best is None or score < best[0]:
        best = (score, list(combo))
VAL_CAMS = best[1]
print('VAL_CAMS =', VAL_CAMS)
summary = pd.DataFrame({'train': cc.drop(index=VAL_CAMS).sum(), 'val': cc.loc[VAL_CAMS].sum()})
summary.index = CLASSES
summary['val share'] = (summary.val / summary.sum(axis=1)).round(3)
# Lab3-style check: class proportions in train vs val
summary['train %'] = (summary.train / summary.train.sum() * 100).round(2)
summary['val %'] = (summary.val / summary.val.sum() * 100).round(2)
n_val_img = meta[(meta.split == 'train') & meta.cam.isin(VAL_CAMS)].shape[0]
print(f'images: train {len(train_files) - n_val_img}, val {n_val_img}')
summary

In [ ]:
if YOLO_DIR.exists():
    shutil.rmtree(YOLO_DIR)
(YOLO_DIR / 'images').mkdir(parents=True); (YOLO_DIR / 'labels').mkdir(parents=True)
clean_by_img = dict(tuple(clean.groupby('image_id')))

def write_sample(src_file, dst_stem):
    shutil.copy(TRAIN_IMG_DIR / src_file, YOLO_DIR / 'images' / f'{dst_stem}.jpg')
    w, h = sizes[src_file]
    g = clean_by_img.get(src_file, empty)
    lines = [f'{r.class_id} {(r.x1 + r.x2) / 2 / w:.6f} {(r.y1 + r.y2) / 2 / h:.6f} {(r.x2 - r.x1) / w:.6f} {(r.y2 - r.y1) / h:.6f}'
             for r in g.itertuples()]
    (YOLO_DIR / 'labels' / f'{dst_stem}.txt').write_text('\n'.join(lines))
    return str(YOLO_DIR / 'images' / f'{dst_stem}.jpg')

# repeat-factor sampling (LVIS-style): r_c = max(1, sqrt(t / f_c)), r_img = max_c r_c
f_c = clean.groupby('class_id').image_id.nunique() / len(train_files)
r_c = np.maximum(1, np.sqrt(RFS_T / f_c)).clip(upper=RFS_MAX)
print('repeat factor per class:', {CLASSES[k]: round(v, 2) for k, v in r_c.items()})
rs = np.random.default_rng(SEED)
def n_repeats(f):
    if not USE_RFS or f not in clean_by_img:
        return 1
    r = r_c[clean_by_img[f].class_id.unique()].max()
    return int(math.floor(r) + (rs.random() < r - math.floor(r)))   # stochastic rounding

lists = {'train': [], 'val': [], 'full': []}
for f in tqdm(train_files):
    cam = f.split('_')[0]
    path = write_sample(f, Path(f).stem)
    reps = [path] + [write_sample(f, f'{Path(f).stem}__rep{k}') for k in range(1, n_repeats(f))]
    lists['full'] += reps
    if cam in VAL_CAMS:
        lists['val'].append(path)          # no repeats in val
    else:
        lists['train'] += reps
for k, v in lists.items():
    (YOLO_DIR / f'{k}.txt').write_text('\n'.join(v))
    print(f'{k}: {len(v)} images (incl. repeats)')

for name, tr in [('data_split.yaml', 'train.txt'), ('data_full.yaml', 'full.txt')]:
    (YOLO_DIR / name).write_text(
        f"path: {YOLO_DIR.as_posix()}\ntrain: {tr}\nval: val.txt\nnc: {len(CLASSES)}\nnames: {CLASSES}\n")
print((YOLO_DIR / 'data_split.yaml').read_text())

## 5. Train

In [ ]:
ModelCls = RTDETR if 'rtdetr' in MODEL else YOLO
if SKIP_TRAIN:
    model = ModelCls(str(WEIGHTS))
else:
    model = ModelCls(MODEL)
    model.train(
        data=str(YOLO_DIR / ('data_full.yaml' if FULL_TRAIN else 'data_split.yaml')),
        imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, patience=PATIENCE, workers=WORKERS,
        seed=SEED, deterministic=True, cos_lr=True, close_mosaic=10, cache='ram',
        project=str(RUNS), name=RUN_NAME, exist_ok=True, plots=True,
        fraction=0.1 if SMOKE_TEST else 1.0,
    )
    # with FULL_TRAIN the val cams are inside training data -> "best" is meaningless, use last epoch
    WEIGHTS = RUNS / RUN_NAME / 'weights' / ('last.pt' if FULL_TRAIN else 'best.pt')
    model = ModelCls(str(WEIGHTS))
print('weights:', WEIGHTS)

In [ ]:
run_dir = RUNS / RUN_NAME
if (run_dir / 'results.csv').exists():
    res = pd.read_csv(run_dir / 'results.csv'); res.columns = res.columns.str.strip()
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    res.plot(x='epoch', y=[c for c in res.columns if c.startswith('train/')], ax=axes[0], title='train losses')
    res.plot(x='epoch', y=[c for c in res.columns if c.startswith('val/')], ax=axes[1], title='val losses')
    res.plot(x='epoch', y=['metrics/mAP50(B)', 'metrics/mAP50-95(B)'], ax=axes[2], title='val mAP')
    plt.tight_layout(); save(f'{RUN_NAME}_curves'); plt.show()

## 6. Evaluate on held-out cameras — pycocotools mAP@50 (same as Kaggle)
Predictions use a very low confidence threshold (`CONF=0.001`): low-confidence boxes only add points at the tail of the precision–recall curve and never reduce AP.

In [ ]:
def predict(model, paths, imgsz=IMGSZ, conf=CONF, tta=TTA, batch=32):
    rows = []
    for i in tqdm(range(0, len(paths), batch), leave=False):
        chunk = [str(p) for p in paths[i:i + batch]]
        for p, r in zip(chunk, model.predict(chunk, imgsz=imgsz, conf=conf, max_det=MAX_DET, augment=tta, verbose=False)):
            b = r.boxes
            for (x1, y1, x2, y2), c, s in zip(b.xyxy.cpu().numpy(), b.cls.cpu().numpy(), b.conf.cpu().numpy()):
                rows.append((Path(p).name, int(c), float(s), float(x1), float(y1), float(x2), float(y2)))
    return pd.DataFrame(rows, columns=['file', 'class_id', 'confidence', 'x1', 'y1', 'x2', 'y2'])

def coco_map50(gt, pred, files, verbose=False):
    # gt: image_id,class_id,x1..y2 ; pred: file,class_id,confidence,x1..y2. Returns (mAP50, per-class AP50)
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    fid = {f: i for i, f in enumerate(files)}
    gt = gt[gt.image_id.isin(fid)]
    ds = {'images': [{'id': i, 'file_name': f, 'width': sizes[f][0], 'height': sizes[f][1]} for f, i in fid.items()],
          'categories': [{'id': k, 'name': n} for k, n in enumerate(CLASSES)],
          'annotations': [{'id': k + 1, 'image_id': fid[r.image_id], 'category_id': int(r.class_id),
                           'bbox': [float(r.x1), float(r.y1), float(r.x2 - r.x1), float(r.y2 - r.y1)],
                           'area': float((r.x2 - r.x1) * (r.y2 - r.y1)), 'iscrowd': 0}
                          for k, r in enumerate(gt.itertuples())]}
    pred = pred[pred.file.isin(fid)]
    dets = [{'image_id': fid[r.file], 'category_id': int(r.class_id), 'score': float(r.confidence),
             'bbox': [float(r.x1), float(r.y1), float(r.x2 - r.x1), float(r.y2 - r.y1)]} for r in pred.itertuples()]
    with (contextlib.nullcontext() if verbose else contextlib.redirect_stdout(io.StringIO())):
        coco_gt = COCO(); coco_gt.dataset = ds; coco_gt.createIndex()
        E = COCOeval(coco_gt, coco_gt.loadRes(dets), 'bbox'); E.evaluate(); E.accumulate(); E.summarize()
    prec = E.eval['precision'][0, :, :, 0, 2]      # IoU=0.50, all recall pts, all classes, area=all, maxDet=100
    per_cls = {CLASSES[k]: (prec[:, k][prec[:, k] > -1].mean() if (prec[:, k] > -1).any() else np.nan) for k in range(len(CLASSES))}
    return E.stats[1], per_cls

val_files = sorted(meta[(meta.split == 'train') & meta.cam.isin(VAL_CAMS)].file)
pred_val = predict(model, [TRAIN_IMG_DIR / f for f in val_files])
val_map50, val_ap = coco_map50(train_df, pred_val, val_files)       # scored against RAW labels
print(f'VAL mAP@50 (held-out cams {VAL_CAMS}) = {val_map50:.4f}')
print('per-class AP@50:', json.dumps({k: round(float(v), 4) for k, v in val_ap.items()}))
ax = pd.Series(val_ap).plot.bar(figsize=(9, 3.5), title=f'Per-class AP@50 (val) — mAP {val_map50:.3f}', color='teal')
ax.bar_label(ax.containers[0], fmt='%.2f'); ax.set_ylim(0, 1); save(f"{RUN_NAME}{'_tta' if TTA else ''}_per_class"); plt.show()

In [ ]:
# error analysis: day vs night, and per validation camera
night_files = set(meta[meta.night].file)
rows = []
for name, fs in [('day', [f for f in val_files if f not in night_files]), ('night', [f for f in val_files if f in night_files])] + \
                [(f'cam {c}', [f for f in val_files if f.startswith(c + '_')]) for c in VAL_CAMS]:
    if fs:
        m, _ = coco_map50(train_df, pred_val, fs)
        rows.append((name, len(fs), round(m, 4)))
pd.DataFrame(rows, columns=['subset', 'images', 'mAP50'])

In [ ]:
from IPython.display import Image as IPImage, display
for fn in ['confusion_matrix_normalized.png', 'BoxPR_curve.png', 'labels.jpg']:
    p = run_dir / fn
    if p.exists():
        print(fn); display(IPImage(str(p), width=800))
        shutil.copy(p, FIG_DIR / f'{RUN_NAME}_{fn}')

## 7. Visualize: ground truth vs prediction (val cameras)

In [ ]:
VIS_CONF = 0.3
pv_by_img = dict(tuple(pred_val.groupby('file')))
pick = list(np.random.default_rng(SEED).choice(val_files, 6, replace=False))
items = []
for f in pick:
    items.append((draw(TRAIN_IMG_DIR / f, gt_by_img.get(f, empty)), f'GT  {f}'))
    items.append((draw(TRAIN_IMG_DIR / f, pv_by_img.get(f, pred_val.iloc[:0]), VIS_CONF), f'PRED (conf>{VIS_CONF})'))
show_grid(items, ncols=4, name=f'{RUN_NAME}_gt_vs_pred')

## 8. Predict test set → submission CSV
**`image_id` = test image file name** (e.g. `1068_20260825_060110.jpg`). The long ids in `sample_submission.csv`
(`dataset_1068_annotated_coco1.0_..._20260825_060110.jpg`) do **not** match the Kaggle solution: a submission using them scored
0.000, while the same predictions with file names scored 0.516. All 1,013 test images are predicted; as required on the Data page,
images without detections get no row and no dummy boxes are added.

In [ ]:
pred_test = predict(model, [TEST_IMG_DIR / f for f in test_files])
sub = pred_test.rename(columns={'file': 'image_id'})
sub = sub.round({'confidence': 5, 'x1': 2, 'y1': 2, 'x2': 2, 'y2': 2})
sub = sub[(sub.x2 > sub.x1) & (sub.y2 > sub.y1)]        # drop degenerate boxes
sub = sub.sort_values(['image_id', 'confidence'], ascending=[True, False]).reset_index(drop=True)
sub.insert(0, 'id', range(len(sub)))
sub = sub[list(sample_sub.columns)]

# sanity checks
assert list(sub.columns) == list(sample_sub.columns)
assert sub.id.is_unique and set(sub.image_id) <= set(test_files)
assert sub.class_id.between(0, 7).all() and (sub.x2 > sub.x1).all() and (sub.y2 > sub.y1).all()
assert (sub.x1 >= 0).all() and (sub.y1 >= 0).all() and (sub.x2 <= 352).all() and (sub.y2 <= 288).all()

tag = RUN_NAME + ('_tta' if TTA else '')
out = SUB_DIR / f'{tag}.csv'
sub.to_csv(out, index=False, encoding='utf-8')
json.dump({'model': MODEL, 'weights': str(WEIGHTS), 'imgsz': IMGSZ, 'epochs': EPOCHS, 'batch': BATCH, 'seed': SEED,
           'full_train': FULL_TRAIN, 'val_cams': VAL_CAMS, 'val_map50': round(float(val_map50), 4), 'rfs': USE_RFS,
           'val_ap': {k: round(float(v), 4) for k, v in val_ap.items()}, 'rfs_t': RFS_T, 'conf': CONF, 'max_det': MAX_DET, 'tta': TTA, 'ultralytics': ultralytics.__version__},
          open(SUB_DIR / f'{tag}.json', 'w'), indent=2)
print(f'saved {out}  rows={len(sub)}  images with detections={sub.image_id.nunique()} / {len(test_files)}')
print('predicted class distribution (conf>0.3):', sub[sub.confidence > 0.3].class_id.map(dict(enumerate(CLASSES))).value_counts().to_dict())
sub.head()

In [ ]:
pt_by_img = dict(tuple(pred_test.groupby('file')))
items = []
for cam, g in meta[meta.split == 'test'].groupby('cam'):
    for f in np.random.default_rng(SEED).choice(g.file.values, 2, replace=False):
        items.append((draw(TEST_IMG_DIR / f, pt_by_img.get(f, pred_test.iloc[:0]), VIS_CONF), f'test cam {cam}'))
show_grid(items, ncols=5, size=3.8, name=f'{RUN_NAME}_test_pred')

## 9. Next experiments
* Bigger model / resolution: `yolo26m.pt`, `IMGSZ=960` (on Colab/Kaggle GPU).
* Compare YOLO26 vs YOLO11 vs RT-DETR on the same val cams; keep the one with the best **val mAP@50**.
* `TTA=True` at inference; ensemble several models with Weighted Boxes Fusion.
* Final submission: `FULL_TRAIN=True` with the best settings (uses all 15 train cameras).
* Submit with Kaggle CLI (max 5/day):
  `kaggle competitions submit -c 2110531-dsde-2026-1 -f submissions/<file>.csv -m "<message>"`